# 1. Data Processing
Reads the raw map-matching dataset (GPS track -> road route), inspects it, cleans it, and writes a clean
intermediate file that `featureEngineering.ipynb` picks up.

Each subject folder has 4 files:

| File | Content |
|---|---|
| `.nodes` | lon, lat of every road-network node (row index = node id) |
| `.arcs` | start node, end node of every directed road arc (row index = arc id) |
| `.track` | lon, lat, timestamp (s) of the noisy GPS trace |
| `.route` | ground-truth sequence of arc ids the vehicle drove |

**Steps:** imports -> config -> loaders -> read -> inspect -> project & build network -> clean tracks -> quality check -> subject-level split -> save

## 1.1 Imports

In [5]:
import os
import pickle

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1.2 Configuration
All thresholds in one place.
- `GAP_S`: a time gap larger than this starts a new track segment.
- `MAX_SPEED`: implied speed above this (m/s, ~216 km/h) is treated as a GPS spike and removed.
- `MIN_COVERAGE`: a subject is flagged low-quality if fewer than this share of its GPS points lie within `ROUTE_R` metres of its own ground-truth route.

In [6]:
DATA_DIR  = "."      # folder that contains train/ and student_test/
TRAIN_DIR = "train"
TEST_DIR  = "student_test"
OUT_DIR   = "processed"

EARTH_R = 6371008.8
GAP_S = 60.0
MAX_SPEED = 60.0
ROUTE_R = 120.0
DENSIFY_M = 15.0
MIN_COVERAGE = 0.80

VAL_FRAC = 0.2        # share of TRAIN subjects held out for validation
SEED = 42
KEEP_BAD = False      # True = keep low-quality train subjects

os.makedirs(os.path.join(OUT_DIR, "interim"), exist_ok=True)

## 1.3 Loader functions
`load_subject` reads the 4 files of one subject. `project` converts lon/lat to local metres
(every subject is a different city, so one local frame per subject; error is <0.1% at city scale).

In [7]:
def load_subject(split_dir, sid):
    p = os.path.join(DATA_DIR, split_dir, sid, sid)
    return dict(
        sid=sid,
        nodes=np.loadtxt(p + ".nodes", ndmin=2),            # (N,2) lon, lat
        arcs=np.loadtxt(p + ".arcs", dtype=int, ndmin=2),   # (A,2) start, end node
        route=np.loadtxt(p + ".route", dtype=int, ndmin=1), # (R,)  arc ids
        track=np.loadtxt(p + ".track", ndmin=2),            # (T,3) lon, lat, t
    )


def project(lon, lat, lon0, lat0):
    x = np.radians(lon - lon0) * EARTH_R * np.cos(np.radians(lat0))
    y = np.radians(lat - lat0) * EARTH_R
    return np.c_[x, y]

## 1.4 Read raw data
Loads every subject of the train and test folders into memory.

In [8]:
def list_subjects(split_dir):
    root = os.path.join(DATA_DIR, split_dir)
    return sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))

raw = {
    "train": {sid: load_subject(TRAIN_DIR, sid) for sid in list_subjects(TRAIN_DIR)},
    "test":  {sid: load_subject(TEST_DIR,  sid) for sid in list_subjects(TEST_DIR)},
}
print({k: len(v) for k, v in raw.items()}, "subjects loaded")

{'train': 80, 'test': 10} subjects loaded


## 1.5 Inspect raw data
One row per subject: sizes, id sanity, timestamp gaps, route continuity, missing values, arc quirks.
This tells us what the cleaning step has to handle.

In [9]:
def inspect_subject(s):
    arcs, route, tr = s["arcs"], s["route"], s["track"]
    dt = np.diff(tr[:, 2])
    contiguous = (np.mean(arcs[route[:-1], 1] == arcs[route[1:], 0]) if len(route) > 1 else 1.0)
    pairs = pd.DataFrame(arcs).duplicated().sum()
    return dict(
        subject_id=s["sid"], n_nodes=len(s["nodes"]), n_arcs=len(arcs), route_len=len(route),
        n_points=len(tr), duration_s=tr[-1, 2] - tr[0, 2], median_dt=np.median(dt), max_gap_s=dt.max(),
        gaps_over_limit=int((dt > GAP_S).sum()), non_increasing_t=int((dt <= 0).sum()),
        nan_values=int(np.isnan(tr).sum()), route_ids_valid=bool(route.max() < len(arcs)),
        route_contiguity=round(float(contiguous), 4), self_loops=int((arcs[:, 0] == arcs[:, 1]).sum()),
        duplicate_arcs=int(pairs),
    )

inspect_df = pd.DataFrame([inspect_subject(s) for split in raw.values() for s in split.values()])
inspect_df["split"] = ["train"] * len(raw["train"]) + ["test"] * len(raw["test"])
inspect_df.groupby("split")[["n_nodes", "n_arcs", "route_len", "n_points", "max_gap_s"]].describe().T.round(1)

split               test     train
n_nodes   count     10.0      80.0
          mean    9068.9   24168.6
          std     7452.2   32272.3
          min     1770.0     299.0
          25%     4908.0    3487.8
          50%     6596.5    9608.0
          75%    11844.2   28846.8
          max    26635.0  155003.0
n_arcs    count     10.0      80.0
          mean   18504.1   48461.3
          std    15496.9   64738.8
          min     3577.0     649.0
          25%    10292.2    7061.8
          50%    13145.0   19526.5
          75%    24818.5   58003.2
          max    56027.0  307392.0
route_len count     10.0      80.0
          mean     347.7     391.7
          std      359.6     361.2
          min       95.0      59.0
          25%      153.5     168.2
          50%      250.0     280.0
          75%      357.8     480.5
          max     1323.0    2266.0
n_points  count     10.0      80.0
          mean    1916.0    2605.2
          std     1335.4    3312.3
          min      262.0     361.0
          25%      917.8     945.8
          50%     1420.0    1529.0
          75%     2872.0    2792.8
          max     3958.0   19313.0
max_gap_s count     10.0      80.0
          mean      25.5    2706.4
          std       65.5   18017.4
          min        1.0       1.0
          25%        1.0       1.8
          50%        2.0      11.0
          75%        2.8      73.8
          max      210.0  153064.0

In [10]:
# summary of potential problems found in the raw data
print("subjects with NaNs            :", int((inspect_df.nan_values > 0).sum()))
print("subjects with non-increasing t:", int((inspect_df.non_increasing_t > 0).sum()))
print("subjects with invalid route id:", int((~inspect_df.route_ids_valid).sum()))
print("subjects with time gaps > %ds :" % GAP_S, int((inspect_df.gaps_over_limit > 0).sum()))
print("subjects with broken route    :", int((inspect_df.route_contiguity < 1).sum()), "(route arcs not end-to-start connected)")
print("total self-loop arcs          :", int(inspect_df.self_loops.sum()))
print("total duplicate arcs          :", int(inspect_df.duplicate_arcs.sum()))
inspect_df.sort_values("max_gap_s", ascending=False).head(5)[["subject_id", "split", "n_points", "duration_s", "max_gap_s"]]

subjects with NaNs            : 0
subjects with non-increasing t: 0
subjects with invalid route id: 0
subjects with time gaps > 60s : 23
subjects with broken route    : 11 (route arcs not end-to-start connected)
total self-loop arcs          : 24
total duplicate arcs          : 1903


,subject_id,split,n_points,duration_s,max_gap_s
72,00000090,train,536,154030.0,153064.0
61,00000078,train,2909,57836.0,52810.0
55,00000069,train,2067,5558.0,2658.0
16,00000021,train,15783,18605.0,2161.0
8,00000011,train,2828,3735.0,909.0


## 1.6 Project coordinates and build the road network
- Nodes/arcs are projected to metres.
- **Arc ids are never renumbered** (the `.route` file refers to them), instead we mark arcs as:
  - `valid` = not a self-loop, has non-zero length, and is the first occurrence of its (start, end) pair
  - `canon_id` = id of the first identical arc, so a route that used a duplicate still matches.

In [11]:
def build_network(s):
    lon0, lat0 = s["nodes"][:, 0].mean(), s["nodes"][:, 1].mean()
    xy = project(s["nodes"][:, 0], s["nodes"][:, 1], lon0, lat0)
    u, v = s["arcs"][:, 0], s["arcs"][:, 1]
    length = np.linalg.norm(xy[v] - xy[u], axis=1)
    canon_id = pd.Series(np.arange(len(u))).groupby([u, v]).transform("min").to_numpy()
    valid = (canon_id == np.arange(len(u))) & (u != v) & (length > 0)
    return dict(lon0=lon0, lat0=lat0, node_xy=xy, arcs=s["arcs"], arc_len=length,
                valid=valid, canon_id=canon_id, route=s["route"])

networks = {sid: build_network(s) for split in raw.values() for sid, s in split.items()}
print("networks built:", len(networks), "| invalid arcs removed from candidate pool:",
      sum(int((~n["valid"]).sum()) for n in networks.values()))

networks built: 90 | invalid arcs removed from candidate pool: 1994


## 1.7 Clean the GPS tracks
Per subject:
1. drop rows with missing values, sort by time, drop duplicate timestamps
2. project to metres in the subject's local frame
3. remove **GPS spikes** (implied speed from the last kept point > `MAX_SPEED`)
4. split into **segments** at time gaps > `GAP_S` and compute `dt` (set to 0 at a segment start)

`point_idx` keeps the original row number so results can always be mapped back to the raw file.

In [12]:
def clean_track(s, net):
    tr = s["track"]
    df = pd.DataFrame({"point_idx": np.arange(len(tr)), "lon": tr[:, 0], "lat": tr[:, 1], "t": tr[:, 2]})
    n_raw = len(df)
    df = df.dropna().sort_values("t").drop_duplicates("t")
    n_dup = n_raw - len(df)

    xy = project(df["lon"].to_numpy(), df["lat"].to_numpy(), net["lon0"], net["lat0"])
    t = df["t"].to_numpy()

    keep = np.ones(len(df), bool)           # spike removal against the last KEPT point
    last = 0
    for i in range(1, len(df)):
        gap = t[i] - t[last]
        if gap <= GAP_S and np.linalg.norm(xy[i] - xy[last]) / max(gap, 1e-6) > MAX_SPEED:
            keep[i] = False
        else:
            last = i
    df, xy, t = df[keep].copy(), xy[keep], t[keep]

    dt = np.r_[np.nan, np.diff(t)]
    new_seg = np.r_[True, dt[1:] > GAP_S]
    dt[new_seg] = 0.0
    df["x"], df["y"] = xy[:, 0], xy[:, 1]
    df["seg_id"] = np.cumsum(new_seg) - 1
    df["dt"] = dt
    df["subject_id"] = s["sid"]
    return df.reset_index(drop=True), dict(dropped_na_or_dup=n_dup, spikes_removed=int((~keep).sum()))

points, clean_log = {"train": [], "test": []}, []
for split in ["train", "test"]:
    for sid, s in raw[split].items():
        p, info = clean_track(s, networks[sid])
        points[split].append(p)
        clean_log.append(dict(subject_id=sid, split=split, n_raw=len(s["track"]), n_clean=len(p),
                              n_segments=int(p.seg_id.max() + 1), **info))
    points[split] = pd.concat(points[split], ignore_index=True)

clean_df = pd.DataFrame(clean_log)
print(clean_df.groupby("split")[["n_raw", "n_clean", "dropped_na_or_dup", "spikes_removed", "n_segments"]].sum())

        n_raw  n_clean  dropped_na_or_dup  spikes_removed  n_segments
split                                                                
test    19160    19158                  0               2          11
train  208417   207595                  0             822         150


## 1.8 Quality check of labels
Some subjects have a ground-truth route that does not match their GPS track
(e.g. the route only covers part of the trace, or the map is offset from the GPS).
Training on those would teach the models wrong labels.

`route_coverage` = share of a subject's GPS points that lie within `ROUTE_R` m of its own true route.
Subjects below `MIN_COVERAGE` are flagged.

In [13]:
def route_coverage(pts, net):
    ids = np.unique(net["canon_id"][net["route"]])
    ids = ids[net["valid"][ids]]
    A, B = net["node_xy"][net["arcs"][ids, 0]], net["node_xy"][net["arcs"][ids, 1]]
    n = np.maximum(np.ceil(np.linalg.norm(B - A, axis=1) / DENSIFY_M).astype(int), 1) + 1
    arc_of = np.repeat(np.arange(len(ids)), n)
    t = (np.arange(n.sum()) - np.repeat(np.cumsum(n) - n, n)) / np.repeat(n - 1, n)
    verts = A[arc_of] + (B[arc_of] - A[arc_of]) * t[:, None]
    d, _ = cKDTree(verts).query(pts[["x", "y"]].to_numpy())
    return float((d <= ROUTE_R).mean())

quality = []
for split in ["train", "test"]:
    for sid, grp in points[split].groupby("subject_id"):
        cov = route_coverage(grp, networks[sid])
        quality.append(dict(subject_id=sid, split=split, route_coverage=round(cov, 4), low_quality=cov < MIN_COVERAGE))
quality_df = pd.DataFrame(quality)
print("flagged subjects:")
quality_df[quality_df.low_quality]

flagged subjects:


,subject_id,split,route_coverage,low_quality
26,00000033,train,0.4961,True
44,00000054,train,0.2320,True


In [14]:
quality_df.groupby("split").route_coverage.describe().round(3)

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,10.0,1.000,0.000,1.000,1.0,1.0,1.0,1.0
train,80.0,0.983,0.102,0.232,1.0,1.0,1.0,1.0


## 1.9 Subject-level train / validation split
`student_test` stays as the test set. The training subjects are split **by subject** (never inside a subject)
into train and validation so there is no leakage between them. Low-quality train subjects are dropped
(only from train/val, never from test) unless `KEEP_BAD = True`.

In [15]:
train_ids = sorted(quality_df[quality_df.split == "train"].subject_id)
bad_ids = sorted(quality_df[(quality_df.split == "train") & quality_df.low_quality].subject_id)
good_ids = train_ids if KEEP_BAD else [s for s in train_ids if s not in bad_ids]

rng = np.random.default_rng(SEED)
val_ids = sorted(rng.choice(good_ids, size=int(round(len(good_ids) * VAL_FRAC)), replace=False).tolist())
fit_ids = [s for s in good_ids if s not in val_ids]
test_ids = sorted(quality_df[quality_df.split == "test"].subject_id)

split_ids = {"train": fit_ids, "val": val_ids, "test": test_ids, "dropped": [] if KEEP_BAD else bad_ids}
print({k: len(v) for k, v in split_ids.items()})
print("dropped:", split_ids["dropped"])

{'train': 62, 'val': 16, 'test': 10, 'dropped': 2}
dropped: ['00000033', '00000054']


## 1.10 Save intermediate output
This is the input of `featureEngineering.ipynb`.

In [16]:
all_points = pd.concat([points["train"], points["test"]], ignore_index=True)
keep_ids = set(fit_ids + val_ids + test_ids)
all_points = all_points[all_points.subject_id.isin(keep_ids)].reset_index(drop=True)

interim = dict(points=all_points, networks={k: v for k, v in networks.items() if k in keep_ids}, split_ids=split_ids,
               config=dict(GAP_S=GAP_S, MAX_SPEED=MAX_SPEED, ROUTE_R=ROUTE_R, MIN_COVERAGE=MIN_COVERAGE, SEED=SEED))
with open(os.path.join(OUT_DIR, "interim", "interim.pkl"), "wb") as f:
    pickle.dump(interim, f)

report = inspect_df.merge(clean_df.drop(columns="split"), on="subject_id").merge(quality_df.drop(columns="split"), on="subject_id")
report.to_csv(os.path.join(OUT_DIR, "subject_report.csv"), index=False)
print("saved", len(all_points), "clean points from", all_points.subject_id.nunique(), "subjects")

saved 201599 clean points from 88 subjects
